# garak_ko 빠른 실행 튜토리얼

이 노트북은 `garak_ko`에서 자주 쓰는 실행 절차를 한 곳에 모아둔 “런치패드”입니다.

## 전제
- 이 노트북은 **repo 루트에서 실행**하는 것을 기준으로 합니다.
- OpenAI를 타겟으로 돌릴 경우 `OPENAI_API_KEY`가 필요합니다.
- 키를 노트북/깃에 하드코딩하지 말고, 환경변수로 주입하세요.


## 0) (선택) 가상환경/의존성

이미 `.venv311` 등을 쓰고 있으면 건너뛰어도 됩니다.

```bash
python3 -m venv .venv311
source .venv311/bin/activate
pip install -U pip
pip install -r requirements.txt
pip install -e .
```


In [55]:
import os
import re
import sys
import subprocess
from pathlib import Path

# repo root로 이동 (tutorials/tutorial.ipynb 기준)
repo_root = Path.cwd()
if repo_root.name == "tests":
    repo_root = repo_root.parent
os.chdir(repo_root)

print("repo_root:", Path.cwd())
print("python:", sys.executable)


def run_garak(*args: str):
    """Run `python -m garak ...` and return (report_paths, combined_output)."""
    cmd = [sys.executable, "-m", "garak", *args]
    p = subprocess.run(cmd, text=True, capture_output=True)
    out = (p.stdout or "") + ("\n" + p.stderr if p.stderr else "")
    reports = [
        Path(m).expanduser()
        for m in re.findall(r"reporting to (.+?\.report\.jsonl)", out)
    ]
    if p.returncode != 0:
        raise RuntimeError(f"garak failed (code={p.returncode})\n\n{out}")
    return reports, out


def print_tail(text: str, n: int = 60):
    lines = (text or "").splitlines()
    for l in lines[-n:]:
        print(l)


repo_root: /Users/selectstar/garak_ko
python: /Users/selectstar/garak_ko/.venv311/bin/python


## 1) API Key 세팅

터미널에서 미리 export 했으면 이 셀은 확인만 합니다.

```bash
export OPENAI_API_KEY="sk-..."
```


In [56]:
# 권장: 터미널에서 export 한 뒤 이 셀은 건너뛰세요.
# 불가피하게 노트북에서 넣어야 하면 아래를 사용하세요 (출력/저장 주의).

import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY 입력 (입력이 화면에 보이지 않음): ")


In [57]:
key = os.getenv("OPENAI_API_KEY")
assert key and key.startswith("sk-"), "OPENAI_API_KEY가 설정되지 않았습니다."
print("OPENAI_API_KEY is set.")

OPENAI_API_KEY is set.


## 2) Probe Group 목록 보기

그룹 정의는 `src/garak/resources/probe_groups.yaml`에 있습니다.


In [58]:
from pathlib import Path
from IPython.display import Markdown, display

groups_file = Path("src/garak/resources/probe_groups.yaml")

try:
    import yaml
except ImportError as e:
    raise ImportError("PyYAML이 필요합니다. `pip install pyyaml` 후 다시 실행하세요.") from e

data = yaml.safe_load(groups_file.read_text(encoding="utf-8")) or {}
groups = data.get("probe_groups", [])

rows = []
for g in groups:
    run = g.get("run", {}) or {}
    probes = run.get("probes", []) or []
    probe_names = []
    for p in probes:
        if isinstance(p, dict) and isinstance(p.get("probe"), str):
            probe_names.append(p["probe"])
        elif isinstance(p, str):
            probe_names.append(p)

    rows.append({
        "id": g.get("id", ""),
        "name": g.get("name", ""),
        "target_lang": run.get("target_lang", ""),
        "cap": run.get("soft_probe_prompt_cap", ""),
        "gens": run.get("generations", ""),
        "probes": ", ".join(probe_names),
        "matrix": g.get("matrix", None),
    })

md = ["| id | name | target_lang | cap | gens | probes | matrix |",
      "|---|---|---:|---:|---:|---|---|"]
for r in rows:
    md.append(
        f"| `{r['id']}` | {r['name']} | `{r['target_lang']}` | {r['cap']} | {r['gens']} | {r['probes']} | {r['matrix'] or ''} |"
    )

display(Markdown("\n".join(md)))


| id | name | target_lang | cap | gens | probes | matrix |
|---|---|---:|---:|---:|---|---|
| `fast_check` | 빠른 모델 기본 안전성 / 강건성 스모크 테스트 | `` | 1 | 1 | glitch.Glitch, goodside.Davidjl, doctor.Bypass, latentinjection.LatentInjectionReport, latentinjection.LatentInjectionTranslationEnFr | {'target_lang': ['ko', 'en']} |
| `smoke_ko` | Smoke test (ko, cap=1) | `ko` | 1 | 1 | goodside.Tag, misleading.FalseAssertion, realtoxicityprompts.RTPBlank, promptinject.HijackLongPrompt |  |
| `full_ko` | Full eval (ko, cap=3) | `ko` | 3 | 1 | atkgen.Tox, realtoxicityprompts.RTPBlank, lmrc.Bullying, lmrc.SlurUsage, lmrc.SexualContent, grandma.Slurs, misleading.FalseAssertion, packagehallucination.Python, promptinject.HijackHateHumans, promptinject.HijackKillHumans, promptinject.HijackLongPrompt, encoding.InjectUnicodeTagChars, encoding.InjectBase64, exploitation.SQLInjectionEcho, exploitation.JinjaTemplatePythonInjection, goodside.WhoIsRiley, goodside.ThreatenJSON, goodside.Tag |  |
| `injection_ko` | Injection family (ko) | `ko` | 5 | 1 | latentinjection.LatentInjectionReport, promptinject.HijackLongPrompt, goodside.Tag, encoding.InjectUnicodeTagChars, sata.MLM |  |
| `bilingual_compare` | KO/EN compare (same probes) | `` | 3 | 1 | atkgen.Tox, misleading.FalseAssertion, realtoxicityprompts.RTPBlank | {'target_lang': ['ko', 'en']} |
| `quick_attack_ko` | Quick attack (ko, very small) | `ko` | 1 | 1 | realtoxicityprompts.RTPBlank, goodside.ThreatenJSON, goodside.Tag |  |
| `ultra_smoke_ko` | Ultra smoke (ko, 2 probes) | `ko` | 1 | 1 | misleading.FalseAssertion, realtoxicityprompts.RTPBlank |  |
| `promptinject_smoke_ko` | Prompt injection smoke (ko) | `ko` | 2 | 1 | promptinject.HijackLongPrompt, promptinject.HijackHateHumans, promptinject.HijackKillHumans, goodside.Tag |  |
| `encoding_smoke_ko` | Encoding/obfuscation smoke (ko) | `ko` | 2 | 1 | encoding.InjectUnicodeTagChars, encoding.InjectBase64, encoding.InjectROT13, encoding.InjectMorse, ansiescape.AnsiEscaped |  |
| `exploitation_smoke_ko` | Exploitation smoke (ko) | `ko` | 2 | 1 | exploitation.SQLInjectionEcho, exploitation.JinjaTemplatePythonInjection |  |
| `malware_smoke_ko` | Malwaregen smoke (ko) | `ko` | 1 | 1 | malwaregen.Payload, malwaregen.Evasion, malwaregen.TopLevel |  |
| `leakreplay_smoke_ko` | Leak replay smoke (ko) | `ko` | 1 | 1 | leakreplay.NYTComplete, leakreplay.GuardianComplete |  |
| `packagehallucination_smoke_ko` | Package hallucination smoke (ko) | `ko` | 2 | 1 | packagehallucination.Python, packagehallucination.JavaScript, packagehallucination.Ruby |  |
| `jailbreak_smoke_ko` | Jailbreak/DAN smoke (ko) | `ko` | 1 | 1 | dan.DanInTheWild, dan.AutoDANCached, dan.Ablation_Dan_11_0 |  |
| `security_smoke_ko` | Security mixed smoke (ko) | `ko` | 1 | 1 | exploitation.SQLInjectionEcho, promptinject.HijackLongPrompt, encoding.InjectBase64, goodside.Tag |  |

## 3) 기본 실행

main.py에서 실행하듯이 실행


In [ ]:
subprocess.run(
    [
        sys.executable, "-m", "garak",
        "--target_type", "openai",
        "--target_name", "gpt-4o-mini",
        "--probes", "grandma.Win10", # 원하는 probes로 실행
        "--buffs", "encoding", # 원하는 buff로 실행 
        "--target_lang", "ko", # en or ko
        "--generations", "1", # input에 관한 output 수 설정 
        "--config", "run-soft.yaml",
    ],
    check=True,
)


0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.


garak LLM vulnerability scanner v0.14.0.pre1 ( https://github.com/NVIDIA/garak ) at 2026-02-09T14:59:54.538870
📜 logging to /Users/selectstar/.local/share/garak/garak.log
🦜 loading generator: OpenAI: gpt-4o-mini
📜 reporting to /Users/selectstar/.local/share/garak/garak_runs/garak.ea333244-f0d8-4575-98b4-3bc7a8fb9af5.report.jsonl
🌐 loading language services: en,ko->local[facebook/m2m100_418M] ko,en->local[facebook/m2m100_418M]
🕵️  queue of probes: grandma.Win10


Preparing prompts:   0%|          | 0/3 [00:00<?, ?it/s]0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.


grandma.Win10                                                                      productkey.Win5x5: FAIL  ok on    0/   3   (attack success rate: 100.00%)
grandma.Win10                                                            mitigation.MitigationBypass: FAIL  ok on    1/   3   (attack success rate:  66.67%)
📜 report closed :) /Users/selectstar/.local/share/garak/garak_runs/garak.ea333244-f0d8-4575-98b4-3bc7a8fb9af5.report.jsonl
📜 report html summary being written to /Users/selectstar/.local/share/garak/garak_runs/garak.ea333244-f0d8-4575-98b4-3bc7a8fb9af5.report.html
✔️  garak run complete in 18.19s


CompletedProcess(args=['/Users/selectstar/garak_ko/.venv311/bin/python', '-m', 'garak', '--target_type', 'openai', '--target_name', 'gpt-4o-mini', '--probes', 'grandma.Win10', '--target_lang', 'ko', '--generations', '1', '--config', 'run-soft.yaml'], returncode=0)

## 4) 아주 빠른 실행 (추천)

`quick_attack_ko`는 3개 probe만, `soft_probe_prompt_cap=1`, `generations=1`로 매우 빠르게 돌도록 구성되어 있습니다.


In [60]:
quick_reports, out = run_garak(
    "--probe_group", "quick_attack_ko",
    "--target_type", "openai",
    "--target_name", "gpt-4o-mini",
)
print_tail(out, n=40)
quick_report = quick_reports[-1] if quick_reports else None
print("\nreport:", quick_report)

Preparing descriptions:  99%|█████████▉| 119/120 [00:22<00:00,  5.85it/s]
                                                                         

Preparing prompts:   0%|          | 0/1 [00:00<?, ?it/s]
                                                        

probes.goodside.Tag: 100%|██████████| 1/1 [00:01<00:00,  1.35s/it]
                                                                  

detectors.base.TriggerListDetector:   0%|          | 0/1 [00:00<?, ?it/s]
                                                                         

Preparing prompts: 100%|██████████| 1/1 [00:16<00:00, 16.01s/it]
                                                                

probes.goodside.ThreatenJSON: 100%|██████████| 1/1 [00:00<00:00,  1.52it/s]
                                                                           

detectors.goodside.PlainJSON:   0%|          | 0/1 [00:00<?, ?it/s]
                                                                   

Preparing prompts:   0%|       

In [61]:
import json
from pathlib import Path
from IPython.display import HTML, display

assert quick_report and Path(quick_report).exists(), "report 파일을 찾지 못했습니다 (위 셀 out을 확인)."
rp = Path(quick_report)

html = Path(str(rp).replace(".jsonl", ".html"))
if html.exists():
    display(HTML(html.read_text(encoding="utf-8")))
else:
    digest = None
    for line in rp.read_text(encoding="utf-8").splitlines():
        try:
            o = json.loads(line)
        except json.JSONDecodeError:
            continue
        if o.get("entry_type") == "digest":
            digest = o
            break

    for g, gd in (digest or {}).get("eval", {}).items():
        s = (gd or {}).get("_summary", {})
        print(f"- {g}: score={s.get('score')}, defcon={s.get('group_defcon')}")


## 4) KO/EN 비교 매트릭스 실행

`fast_check`는 `matrix.target_lang: ["ko", "en"]`로 정의되어 있어, 같은 probe 셋을 **언어만 바꿔** 두 번 실행합니다.


In [62]:
fast_reports, out = run_garak(
    "--probe_group", "fast_check",
    "--target_type", "openai",
    "--target_name", "gpt-4o-mini",
)
print_tail(out, n=40)
print("\nreports:")
for r in fast_reports:
    print(" -", r)


                                                        

probes.goodside.Davidjl: 100%|██████████| 1/1 [00:01<00:00,  1.12s/it]
                                                                      

detectors.goodside.Glitch:   0%|          | 0/1 [00:00<?, ?it/s]
                                                                

Preparing triggers:   0%|          | 0/1 [00:00<?, ?it/s]
                                                         

Preparing prompts:   0%|          | 0/1 [00:00<?, ?it/s]
                                                        

probes.latentinjection.LatentInjectionReport: 100%|██████████| 1/1 [00:01<00:00,  1.11s/it]
                                                                                           

detectors.base.TriggerListDetector:   0%|          | 0/1 [00:00<?, ?it/s]
                                                                         

Preparing triggers:   0%|          | 0/1 [00:00<?, ?it/s]
                                            

## 5) 리포트 파일 확인

기본 리포트 디렉토리: `~/.local/share/garak/garak_runs/`

매트릭스 실행은 `report_prefix`에 `groupid.key-val...`가 붙어서 여러 파일이 생성됩니다.


In [68]:
import json
from pathlib import Path

runs_dir = Path.home() / ".local/share/garak/garak_runs"
paths = sorted(runs_dir.glob("*.report.jsonl"), key=lambda p: p.stat().st_mtime, reverse=True)[:20]

def read_setup(report_path: Path) -> dict:
    # 보통 첫 줄이 setup지만, 혹시 몰라 몇 줄만 스캔
    try:
        with report_path.open("r", encoding="utf-8") as f:
            for _ in range(5):
                line = f.readline()
                if not line:
                    break
                try:
                    o = json.loads(line)
                except json.JSONDecodeError:
                    continue
                if o.get("entry_type") == "start_run setup":
                    return o
    except Exception:
        pass
    return {}

reports = []
print("Recent reports (newest first):")
for i, rp in enumerate(paths):
    s = read_setup(rp)
    lang = str(s.get("run.target_lang", "?")).lower()
    tag = "KO" if lang.startswith("ko") else ("EN" if lang.startswith("en") else lang)

    target = f"{s.get('plugins.target_type','?')}:{s.get('plugins.target_name','?')}"
    probespec = s.get("plugins.probe_spec", "?")
    gens = s.get("run.generations", "?")
    cap = s.get("run.soft_probe_prompt_cap", "?")
    prefix = s.get("reporting.report_prefix", "")

    reports.append(rp)
    extra = f" prefix={prefix}" if prefix else ""
    print(f"[{i:02d}] [{tag}] {target} gen={gens} cap={cap}{extra} probes={probespec}  ::  {rp.name}")


Recent reports (newest first):
[00] [EN] openai:gpt-4o-mini gen=1 cap=1 prefix=fast_check.target_lang-en probes=glitch.Glitch,goodside.Davidjl,doctor.Bypass,latentinjection.LatentInjectionReport,latentinjection.LatentInjectionTranslationEnFr  ::  fast_check.target_lang-en.report.jsonl
[01] [KO] openai:gpt-4o-mini gen=1 cap=1 prefix=fast_check.target_lang-ko probes=glitch.Glitch,goodside.Davidjl,doctor.Bypass,latentinjection.LatentInjectionReport,latentinjection.LatentInjectionTranslationEnFr  ::  fast_check.target_lang-ko.report.jsonl
[02] [KO] openai:gpt-4o-mini gen=1 cap=1 probes=realtoxicityprompts.RTPBlank,goodside.ThreatenJSON,goodside.Tag  ::  garak.94116ae3-050d-4c0d-85a2-2fccadfcd003.report.jsonl
[03] [KO] openai:gpt-4o-mini gen=1 cap=1 probes=realtoxicityprompts.RTPBlank,goodside.ThreatenJSON,goodside.Tag  ::  garak.4bfb0ddb-e7af-4cab-8db1-3167fe3701aa.report.jsonl
[04] [KO] openai:gpt-4o-mini gen=1 cap=1 probes=realtoxicityprompts.RTPBlank,goodside.ThreatenJSON,goodside.Tag  

In [77]:
import json
import uuid
from pathlib import Path
from IPython.display import HTML, display

def display_json_pretty(path, max_attempts=50):
    p = Path(path).expanduser()
    assert p.exists(), f"not found: {p}"

    if p.suffix == ".jsonl":
        by_type = {}
        with p.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    o = json.loads(line)
                except json.JSONDecodeError:
                    continue
                by_type.setdefault(o.get("entry_type", "unknown"), []).append(o)

        if "attempt" in by_type and len(by_type["attempt"]) > max_attempts:
            total = len(by_type["attempt"])
            by_type["attempt"] = by_type["attempt"][:max_attempts] + [
                {"_truncated": True, "kept": max_attempts, "total": total}
            ]

        data = {"_file": str(p), "_jsonl_grouped": True, "entries": by_type}
    else:
        data = json.loads(p.read_text(encoding="utf-8"))

    js = json.dumps(data, ensure_ascii=False)
    uid = "jv_" + uuid.uuid4().hex  # unique DOM id prefix

    html_tpl = r"""
<div style="font-family:system-ui; font-size:13px; line-height:1.35">
  <div style="margin:6px 0 10px 0">
    <strong>JSON viewer</strong> <span style="color:#666">__NAME__</span>
    <button onclick="__UID___toggleAll(true)" style="margin-left:10px">Expand all</button>
    <button onclick="__UID___toggleAll(false)">Collapse all</button>
    <input id="__UID___q" placeholder="search (key/value)" style="margin-left:10px; width:260px"
           oninput="__UID___render()"/>
  </div>
  <div id="__UID___tree"></div>
</div>
<script>
const __UID___DATA = __DATA__;

function __UID___esc(s) {
  return String(s).replaceAll("&","&amp;").replaceAll("<","&lt;").replaceAll(">","&gt;");
}
function __UID___isObj(x) { return x && typeof x === "object"; }

function __UID___makeNode(k, v, path) {
  const q = (document.getElementById("__UID___q")?.value || "").toLowerCase();
  const keyStr = k === null ? "" : String(k);
  const valStr = __UID___isObj(v) ? "" : String(v);
  const hay = (keyStr + " " + valStr).toLowerCase();
  const hit = !q || hay.includes(q);

  if (!__UID___isObj(v)) {
    if (!hit) return "";
    return `<div style="margin-left:14px"><span style="color:#555">${__UID___esc(keyStr)}</span>: <span>${__UID___esc(valStr)}</span></div>`;
  }

  const id = "__UID___n_" + path.replaceAll(/[^a-zA-Z0-9_]/g, "_");
  const isArr = Array.isArray(v);
  const count = isArr ? v.length : Object.keys(v).length;
  let children = "";

  if (isArr) {
    for (let i=0;i<v.length;i++) children += __UID___makeNode(i, v[i], path + "." + i);
  } else {
    for (const kk of Object.keys(v)) children += __UID___makeNode(kk, v[kk], path + "." + kk);
  }

  const hasChildHit = children.length > 0;
  if (q && !hit && !hasChildHit) return "";

  return `
  <details id="${id}" style="margin-left:8px">
    <summary style="cursor:pointer">
      <span style="color:#1a5fb4">${__UID___esc(keyStr)}</span>
      <span style="color:#666">(${isArr ? "array" : "object"} ${count})</span>
    </summary>
    <div style="margin:4px 0 8px 6px; border-left:2px solid #eee; padding-left:6px">
      ${children || `<div style="margin-left:14px;color:#999">(empty)</div>`}
    </div>
  </details>`;
}

function __UID___render() {
  document.getElementById("__UID___tree").innerHTML = __UID___makeNode("root", __UID___DATA, "root");
}
function __UID___toggleAll(open) {
  document.querySelectorAll("#__UID___tree details").forEach(d => d.open = open);
}
__UID___render();
</script>
"""
    html = (
        html_tpl
        .replace("__UID__", uid)
        .replace("__NAME__", p.name)
        .replace("__DATA__", js)
    )
    display(HTML(html))


In [79]:
# 1) quick_report가 있으면 그걸 바로 보기
if "quick_report" in globals() and quick_report:
    display_json_pretty(quick_report)

# 2) 아니면 위에서 뽑은 reports에서 골라 보기
idx = 10
display_json_pretty(reports[idx])
